# AgentCore Gateway를 통해 Agent를 Coinbase Bazaar와 통합

## 개요

**Coinbase x402 Bazaar**는 유료 tool이 semantic description, 가격, input/output schema와 함께 등록된 MCP marketplace입니다. Agent는 `search_resources`를 통해 tool을 검색하고 `proxy_tool_call`을 통해 호출하며, Bazaar가 402 감지와 payment routing을 처리합니다.

이 튜토리얼에서는 Bazaar를 native MCP target으로 **AgentCore Gateway**에 연결한 다음, 유료 tool을 검색하고 자동 payment 처리와 함께 호출하는 Strands agent를 구축합니다.

### 작동 방식

```
┌─────────────────────────────────┐
│  🧑‍💻 Developer Code              │
│                                 │
│  Strands Agent                  │
│  + AgentCorePaymentsPlugin      │
│  + MCPClient (streamable HTTP)  │
└──────────┬──────────────────────┘
           │ MCP protocol
┌──────────▼──────────────────────┐
│  🔀 AgentCore Gateway            │
│  Target: Coinbase x402 Bazaar   │
│  (No outbound auth)             │
└──────────┬──────────────────────┘
           │
┌──────────▼──────────────────────┐
│  🌐 Coinbase x402 Bazaar        │
│  search_resources → discover    │
│  proxy_tool_call  → call + pay  │
└──────────┬──────────────────────┘
           │ HTTP 402 → pay → retry
┌──────────▼──────────────────────┐   ┌──────────────────┐
│  ☁️ AgentCore payments           │──▶│ 🏦 Wallet Provider│
│  Payment Manager                │   │ Coinbase CDP     │
│  Session ($1.00 budget)         │   │   — or —         │
│  Instrument (embedded wallet)   │   │ Stripe Privy     │
│  ProcessPayment (sign + proof)  │   │ (routed by       │
│                                 │   │  PaymentConnector)│
└─────────────────────────────────┘   └──────────────────┘
```

Tutorial 01과의 핵심 차이점은 agent가 호출할 URL을 미리 알지 못한다는 것입니다. Runtime에 `search_resources`로 tool을 검색한 다음 `proxy_tool_call`로 호출합니다. Payment infrastructure는 동일합니다.

### Wallet provider에 독립적인 설계

Tutorial 00에서 Coinbase CDP 또는 Stripe(Privy) 중 어느 것을 구성했든 이 Notebook의 agent 코드는 동일합니다. `AgentCorePaymentsPlugin`은 `payment_instrument_id`를 받으며, AgentCore payments는 PaymentConnector를 기반으로 해당 instrument를 지원하는 wallet provider를 식별합니다. 코드, Gateway, Bazaar tool은 동일하고 `.env` 값만 다릅니다. 이는 개발자가 payment logic을 한 번만 작성하면 service가 provider routing을 처리하는 AgentCore payments의 핵심 설계 원칙입니다.

> **Testnet 전용입니다.** 이 튜토리얼에서는 [faucet.circle.com](https://faucet.circle.com/)의 무료 USDC와 함께 Base Sepolia(Ethereum)를 사용합니다. Testnet USDC에는 실제 가치가 없습니다.


### 아키텍처 개요

![Architecture](images/architecture.png)


## 사전 요구 사항

* [Tutorial 00 — AgentCore payments 설정](../00-setup-agentcore-payments/) 완료(`.env`에 payment manager, instrument, session 존재)
* Wallet에 testnet USDC 입금 완료 — [Circle USDC Faucet 안내](https://faucet.circle.com/) 참조(Base Sepolia를 선택하고 Tutorial 00의 wallet address 입력)
* AgentCore CLI: `npm install -g @aws/agentcore`
* `pip install -r requirements.txt`

이 튜토리얼은 Tutorial 00에서 구성한 Coinbase CDP 또는 Stripe(Privy) wallet provider 모두에서 작동합니다. 선택한 provider와 관계없이 agent 코드는 동일합니다.

AWS credentials에는 Tutorial 00에서 생성한 IAM 권한(`setup_payment_roles()`)이 필요합니다. Tutorial 00을 성공적으로 실행했다면 필요한 권한이 이미 있습니다.

In [ ]:
%pip install -r requirements.txt --quiet

In [ ]:
import os
# os.environ['AWS_PROFILE'] = 'your-profile'

import boto3

session = boto3.Session()
identity = session.client("sts").get_caller_identity()
print(f"Authenticated as: {identity['Arn']}")
print(f"Account: {identity['Account']}")
print(f"Region: {session.region_name}")

## 1단계 — Bazaar Target이 있는 Gateway 생성

> **비용 안내:** AgentCore Gateway 배포에는 AWS 요금이 발생합니다. 지속적인 비용을 방지하려면 작업을 마친 후 리소스를 정리하세요.

Coinbase x402 Bazaar를 AgentCore Gateway의 MCP target으로 추가합니다. Bazaar는 agent가 10,000개 이상의 x402 지원 service를 탐색하고 검색할 수 있는 discovery endpoint를 제공합니다.

### 옵션 A: AgentCore Console(권장)

1. [Amazon Bedrock AgentCore console](https://console.aws.amazon.com/bedrock-agentcore/)을 엽니다.
2. Gateway → Create Gateway → Add Target으로 이동합니다.
3. Target type: **Integrations**
4. **Coinbase x402 Bazaar**를 선택합니다.
5. Outbound auth는 필요하지 않습니다(No Authorization이 기본값).

### 옵션 B: AgentCore CLI

> **참고:** AgentCore CLI는 Coinbase x402 Bazaar를 자동 구성하는 **Integrations** target type을 지원하지 않습니다. CLI에서는 Bazaar endpoint URL을 직접 사용하여 표준 `mcp-server` target으로 추가합니다. 결과는 동일하며, Bazaar MCP server가 Gateway를 통해 노출됩니다.

```bash
# 1. AgentCore 프로젝트 생성(Tutorial 02에서 이미 생성했다면 건너뜀)
agentcore create --name BazaarAgent --defaults

# 2. Gateway 추가
agentcore add gateway --name BazaarGateway

# 3. Coinbase x402 Bazaar를 MCP 대상으로 추가
agentcore add gateway-target \
  --name CoinbaseBazaar \
  --type mcp-server \
  --endpoint https://api.cdp.coinbase.com/platform/v2/x402/discovery/mcp \
  --gateway BazaarGateway

# 4. Gateway 배포
agentcore deploy -y

# 5. Gateway URL과 인증 정보 확인
agentcore fetch access --name BazaarGateway --type gateway
```

출력의 `GATEWAY_URL`을 저장하고 `.env` 파일에 추가합니다.

```bash
# 00-getting-started/.env 파일에 추가:
GATEWAY_URL=https://<gateway-id>.gateway.bedrock-agentcore.<region>.amazonaws.com/mcp

# Gateway가 CUSTOM_JWT 인증을 사용한다면 다음 항목도 추가:
CLIENT_ID=<cognito-client-id>
CLIENT_SECRET=<cognito-client-secret>
TOKEN_URL=https://<domain>.auth.<region>.amazoncognito.com/oauth2/token
```

`agentcore fetch access` 출력에서 URL과 Gateway에 필요한 auth를 확인할 수 있습니다. `--authorizer-type` flag 없이 `agentcore add gateway`를 사용했다면 기본값은 NONE auth이므로 `GATEWAY_URL`만 필요합니다.

Gateway URL을 확인한 후 아래 셀에 붙여 넣어 `.env`에 저장합니다.

In [ ]:
# console에서 생성한 Gateway URL을 여기에 붙여 넣기
import sys

sys.path.append("..")
from utils import update_env_file

# GATEWAY_URL = 'https://<your-gateway-id>.gateway.bedrock-agentcore.<region>.amazonaws.com/mcp'
GATEWAY_URL = input("Enter your Gateway URL: ").strip()

if not GATEWAY_URL:
    print("No URL provided. Edit the GATEWAY_URL variable above or re-run this cell.")
else:
    update_env_file("../.env", {"GATEWAY_URL": GATEWAY_URL})
    os.environ["GATEWAY_URL"] = GATEWAY_URL
    print(f"Saved GATEWAY_URL to .env: {GATEWAY_URL}")

## 2단계 — Payment Config 불러오기

이 단계에서는 [Tutorial 00](../00-setup-agentcore-payments/)에서 생성한 payment resource를 `.env`에서 불러옵니다. 모든 튜토리얼(01~06)에서 동일한 config loading pattern을 사용하며 여기서는 새 resource를 생성하지 않습니다.

### 2a단계 — Payment Config(모든 튜토리얼에서 동일)

In [ ]:
import sys
import os

sys.path.append(os.path.join(os.path.dirname(os.path.abspath(".")), ""))
sys.path.append("..")

from dotenv import load_dotenv

env_path = os.path.join(os.path.dirname(os.path.abspath(".")), ".env")
if not os.path.exists(env_path):
    env_path = os.path.join("..", ".env")
load_dotenv(dotenv_path=env_path, override=True)
print(f"Loaded .env from: {env_path}")

from utils import load_tutorial_env, print_summary

config = load_tutorial_env()
PAYMENT_MANAGER_ARN = config["payment_manager_arn"]
REGION = config["region"]
USER_ID = config["user_id"]

if config.get("multi_provider"):
    PROVIDER = list(config["instruments"].keys())[0]
    INSTRUMENT_ID = config["instruments"][PROVIDER]["instrument_id"]
else:
    INSTRUMENT_ID = config["instrument_id"]
    PROVIDER = config.get("provider_type", "unknown")

from bedrock_agentcore.payments import PaymentManager

manager = PaymentManager(payment_manager_arn=PAYMENT_MANAGER_ARN, region_name=REGION)
instr = manager.get_payment_instrument(user_id=USER_ID, payment_instrument_id=INSTRUMENT_ID)
instr_status = instr.get("status", "UNKNOWN")
assert instr_status == "ACTIVE", f"Instrument is {instr_status} — fund and delegate in Tutorial 00/03 first"

print_summary(
    "Payment Config",
    manager_arn=PAYMENT_MANAGER_ARN,
    region=REGION,
    provider=PROVIDER,
    instrument_id=INSTRUMENT_ID,
    instrument_status=instr_status,
    gateway_url=os.environ.get("GATEWAY_URL", "NOT SET"),
)

### 2b단계 — Gateway Config

In [ ]:
import os

# Gateway URL — 1단계에서 Gateway를 생성한 후 설정
GATEWAY_URL = os.environ.get("GATEWAY_URL", "")
if not GATEWAY_URL:
    raise ValueError("GATEWAY_URL not set. Create the Gateway in Step 1, then add GATEWAY_URL to your .env file.")

print_summary(
    "Gateway Config",
    gateway_url=GATEWAY_URL,
)

## 3단계 — Payment Session 생성

Payment session은 이 interaction의 budget과 expiry를 정의합니다. Tutorial 00의 Payment Manager와 Instrument는 장기 유지 resource이며, Session은 필요할 때 생성합니다.

In [ ]:
# 이 task를 위한 새 session 생성(2단계의 manager SDK client 사용)
sess_resp = manager.create_payment_session(
    user_id=USER_ID,
    limits={"maxSpendAmount": {"value": "1.00", "currency": "USD"}},
    expiry_time_in_minutes=60,
)
SESSION_ID = sess_resp["paymentSessionId"]
print(f"✅ Session: {SESSION_ID} (budget: $1.00)")

## 4단계 — Gateway 연결 및 Agent 생성

Strands MCP client를 사용하여 Gateway MCP endpoint에 연결합니다. Gateway는 `search_resources`와 `proxy_tool_call`을 MCP tool로 노출하는 Bazaar로 요청을 routing합니다.

`proxy_tool_call`이 402를 반환하면 PaymentsPlugin이 x402 payment를 자동으로 처리합니다.

### Request 및 Payment 순서

![Payment Sequence](images/payment_sequence.png)


In [ ]:
from datetime import timedelta
from mcp.client.streamable_http import streamablehttp_client
from strands import Agent
from strands.models import BedrockModel
from strands.tools.mcp.mcp_client import MCPClient
from bedrock_agentcore.payments.integrations.strands import (
    AgentCorePaymentsPlugin,
    AgentCorePaymentsPluginConfig,
)

# Gateway auth — .env에서 자동 감지
# Gateway에서 CUSTOM_JWT를 사용하면 .env에 CLIENT_ID, CLIENT_SECRET, TOKEN_URL 설정
# 기본값인 NONE auth를 사용하면 설정하지 않음
# `agentcore fetch access`를 실행하여 Gateway에 필요한 auth 확인
gateway_headers = {}
CLIENT_ID = os.environ.get("CLIENT_ID")
CLIENT_SECRET = os.environ.get("CLIENT_SECRET")
TOKEN_URL = os.environ.get("TOKEN_URL")

if CLIENT_ID and CLIENT_SECRET and TOKEN_URL:
    from utils import get_oauth_token

    token = get_oauth_token(TOKEN_URL, CLIENT_ID, CLIENT_SECRET)
    gateway_headers = {"Authorization": f"Bearer {token}"}
    print("Gateway auth: CUSTOM_JWT (OAuth token acquired)")
else:
    print("Gateway auth: NONE (no CLIENT_ID/CLIENT_SECRET/TOKEN_URL in .env)")

# Gateway MCP endpoint에 연결
mcp_client = MCPClient(
    lambda: streamablehttp_client(
        GATEWAY_URL,
        headers=gateway_headers,
        timeout=timedelta(seconds=120),
    )
)

# Payment plugin — Payment Manager, Instrument, Session 사용
# Plugin이 x402 402 response를 자동 처리: intercept → sign → retry
# 수동 payment 코드 불필요
payment_plugin = AgentCorePaymentsPlugin(
    config=AgentCorePaymentsPluginConfig(
        payment_manager_arn=PAYMENT_MANAGER_ARN,
        user_id=USER_ID,
        payment_instrument_id=INSTRUMENT_ID,
        payment_session_id=SESSION_ID,
        region=REGION,
        network_preferences_config=["eip155:84532", "base-sepolia"],
    )
)

SYSTEM_PROMPT = """You are a research agent with access to the Coinbase x402 Bazaar — a marketplace of paid tools.

You can:
1. Use search_resources to discover available paid tools (filter by network, query, etc.)
2. Use proxy_tool_call to call a discovered tool — payment is handled automatically

When asked to find information:
- First search for relevant tools on the Bazaar
- Then call the most relevant tool
- Report what you found and what it cost

Always be transparent about payments."""

# MCP connection 열기(리소스를 정리할 때까지 셀 간에 유지)
mcp_client.__enter__()

agent = Agent(
    model=BedrockModel(model_id="us.anthropic.claude-sonnet-4-6", streaming=True),
    tools=mcp_client.list_tools_sync(),
    plugins=[payment_plugin],
    system_prompt=SYSTEM_PROMPT,
)
print(f"✅ Agent created with {len(mcp_client.list_tools_sync())} Bazaar tools + payment plugin")

## 5단계 — 유료 Tool 검색 및 호출

이 부분이 [Tutorial 01](../01-agents-payments-and-limits/)과의 핵심 차이점입니다. Tutorial 01에서는 agent에 호출할 URL을 지정했습니다. 여기서는 agent가 `search_resources`를 사용하여 호출 대상을 직접 검색한 다음 `proxy_tool_call`로 결제하고 호출합니다. 개발자가 존재하는 tool을 미리 구성하지 않으며 agent가 runtime에 검색합니다.

문서의 사용 사례인 유료 data source에 액세스하는 research agent, market data에 액세스하는 financial analysis agent, 유료 content를 탐색하는 agent는 모두 검색에서 시작합니다.

### 5a단계 — 유료 Tool 검색 및 호출

In [ ]:
result = agent(
    "Search the Bazaar for paid data sources related to market news on Base Sepolia. "
    "Tell me what tools are available, their prices, and what data they provide. "
    "Then pick the most relevant one, call it, and summarize the results with the cost."
)
print(result.message)

### 5b단계 — Multi-Tool Discovery: Category 간 가격 비교

Agent는 여러 category를 검색하고 가격을 비교하여 호출 대상을 결정하기 전에 여러 유료 service를 평가할 수 있음을 보여 줍니다. 이는 10,000개 이상의 pay-per-use x402 endpoint 중 agent가 budget 내에서 최적의 옵션을 찾는 endpoint discoverability 기능에 해당합니다.

In [ ]:
result = agent(
    "Search the Bazaar for three different categories of paid tools on Base Sepolia: "
    "1) market news, 2) weather data, "
    "For each category, list the available tools with their prices. "
    "Then tell me which tool in each category is the cheapest."
)
print(result.message)

### 5c단계 — Budget을 고려한 Tool 선택

Agent는 비용이 높은 tool을 호출하기 전에 남은 budget을 확인합니다. Budget이 적으면 더 저렴한 대안을 선택합니다. 이를 통해 payment limit과 tool discovery가 함께 작동하여 agent가 runtime에 비용을 고려한 결정을 내리는 방식을 확인할 수 있습니다.

In [ ]:
# 현재 지출 확인
mid_session = manager.get_payment_session(
    user_id=USER_ID,
    payment_session_id=SESSION_ID,
)
current = mid_session.get("availableLimits", {}).get("availableSpendAmount", {})
print(f"Remaining budget: {current}")

result = agent(
    f"My remaining budget is {current} out of a $1.00 budget. "
    "Search the Bazaar for tools under $0.10 on Base Sepolia. "
    "Pick the cheapest one and call it. "
    "If nothing is under $0.10, tell me what the cheapest option costs."
)
print(result.message)

### 5d단계 — 단일 Session에서 여러 Bazaar 호출

Agent는 단일 session에서 여러 유료 tool을 순차적으로 호출합니다. Session budget이 모든 호출의 누적 지출을 추적하므로 하나의 payment stack이 provider별 설정 없이 여러 merchant를 처리함을 보여 줍니다.

In [ ]:
result = agent(
    "I want a comprehensive research report. Do the following in order:\n"
    "1. Search the Bazaar for a market news tool and call it for the latest market updates\n"
    "2. Search for a weather data tool and call it for San Francisco weather\n"
    "3. Search for a crypto news tool and call it for the latest crypto headlines\n"
    "After each call, note the cost. At the end, summarize all results "
    "and the total amount spent across all three calls."
)
print(result.message)

## 6단계 — Session 지출 확인

In [ ]:
session_info = manager.get_payment_session(
    user_id=USER_ID,
    payment_session_id=SESSION_ID,
)
sess = session_info
available = sess.get("availableLimits", {}).get("availableSpendAmount", {})
budget = sess.get("limits", {}).get("maxSpendAmount", {})
spent = float(budget.get("value", 0)) - float(available.get("value", budget.get("value", 0)))

print_summary(
    "Session After Bazaar Calls",
    session_id=SESSION_ID,
    budget_limit=f"${budget.get('value', 'N/A')} {budget.get('currency', '')}",
    remaining=f"${available.get('value', 'N/A')} {available.get('currency', '')}",
    spent=f"${spent:.4f} USD",
)

## Payment Trace 보기

Payment를 발생시킨 모든 Bazaar tool 호출에서 trace가 생성됩니다. CloudWatch GenAI Observability Dashboard에서 확인하세요.


In [ ]:
print("🔍 View your agent traces: CloudWatch → GenAI Observability Dashboard")
print(f"  https://{REGION}.console.aws.amazon.com/cloudwatch/home?region={REGION}#gen-ai-observability/agent-core")

## 요약

Build 시점에 알지 못했던 tool을 검색하고 결제하는 agent를 구축했습니다. 이는 AgentCore Gateway를 통해 10,000개 이상의 pay-per-use x402 endpoint를 노출하는 즉시 사용 가능한 Coinbase x402 Bazaar MCP server인 **endpoint discoverability** 기능을 보여 줍니다.

| AgentCore payments 기능 | 이 튜토리얼에서 확인한 내용 |
|---------------------------|-------------------------------|
| Endpoint discoverability | Agent가 하나의 Gateway URL에 연결하고 runtime에 market news, weather data, crypto news category의 tool 검색 |
| Payment processing | 개발자의 payment 코드 없이 모든 Bazaar tool 호출에 대해 `AgentCorePaymentsPlugin`이 402 → sign → retry 처리 |
| Payment limits | Session budget이 서로 다른 merchant의 여러 유료 tool 호출에 대한 누적 지출 추적 |
| Wallet integration | 동일한 Notebook, agent 코드, Gateway가 Coinbase CDP 또는 Stripe(Privy)에서 작동하며 Tutorial 00의 `.env` 값만 다름 |

Tutorial 01에서는 agent에 호출할 URL을 지정했지만 여기서는 agent가 task를 기반으로 호출 대상을 결정합니다. 이는 문서의 Research 및 Financial analysis 사용 사례에 해당합니다.

### Wallet provider에 독립적인 설계

이는 AgentCore payments의 핵심 설계 원칙입니다. Agent 코드는 Coinbase 또는 Privy를 참조하지 않습니다. Plugin config에서 `payment_instrument_id`를 받으면 AgentCore payments가 Tutorial 00에서 구성한 PaymentConnector를 기반으로 해당 instrument를 지원하는 wallet provider를 식별합니다. 동일하게 배포된 agent, Gateway, Bazaar tool이 코드 변경 없이 두 wallet provider를 모두 지원합니다. 개발자가 Coinbase에서 Privy로 전환하거나 둘 다 추가할 때는 `.env`만 변경하면 되며 Notebook 또는 agent 코드는 변경할 필요가 없습니다.

### 배포된 Agent의 Role 분리

이 Notebook은 AWS credentials로 로컬에서 실행됩니다. 배포 시 runtime process는 ProcessPaymentRole로 실행되며, plugin은 app backend에서 설정한 budget 내에서 agent를 대신해 `ProcessPayment`를 호출합니다. Runtime은 session 생성, limit 수정 또는 wallet provision을 할 수 없습니다. Agent(LLM)는 `ProcessPayment`를 직접 호출하지 않습니다. 전체 role 분리 구현은 Tutorial 02를 참조하세요.

Role 분리를 로컬에서 테스트하려면 assumed-role session을 SDK client에 전달합니다.

```python
from utils import assume_role
import boto3

# 앱 백엔드(ManagementRole)가 세션 생성
manager = PaymentManager(payment_manager_arn=ARN, region_name=REGION)
session = manager.create_payment_session(user_id=USER_ID, ...)

# Agent는 ProcessPaymentRole로 실행되며 ProcessPayment만 수행 가능
agent_session = assume_role(boto3.Session(), PROCESS_PAYMENT_ROLE_ARN, 'agent')
agent_manager = PaymentManager(
    payment_manager_arn=ARN, boto3_session=agent_session
)
# Plugin에 agent_manager 전달 - 세션을 생성하거나 예산을 수정할 수 없음
```

In [ ]:
# MCP connection 종료
try:
    mcp_client.__exit__(None, None, None)
    print("MCP connection closed.")
except Exception:
    pass

## 리소스 정리(선택 사항)

Session은 `expiryTimeInMinutes`가 지나면 자동으로 만료됩니다.

**Tutorial 05~06을 계속 진행할 계획이라면 리소스를 정리하지 마세요.** Gateway와 payment resource는 여러 튜토리얼에서 재사용됩니다.

모든 튜토리얼을 마친 후에만 리소스를 정리하세요.

```bash
# Gateway만 제거(결제 리소스와 배포된 agent는 유지)
# agentcore remove gateway --name BazaarGateway -y
# agentcore remove gateway-target --name CoinbaseBazaar -y
# agentcore deploy -y

# 또는 AgentCore 프로젝트의 모든 항목 제거
# agentcore remove all -y

# 결제 리소스(Manager, Connector, Instrument): Tutorial 00의 cleanup 실행
```


# 축하합니다!

AgentCore Gateway를 통해 Coinbase Bazaar의 tool을 검색하고 결제하는 agent를 구축했습니다.

### 다음 단계

- **Custom Lambda interceptor** — 요청이 Coinbase Bazaar에 도달하기 전에 변환, 필터링 또는 보강하도록 Lambda 기반 interceptor를 AgentCore Gateway에 추가
- **AgentCore Runtime에 배포** — Tutorial 02 pattern에 따라 ProcessPaymentRole 적용과 함께 이 agent 배포
- **Gateway를 사용하는 Multi-agent** — 동일한 Gateway target에 대해 독립적인 budget으로 여러 agent를 실행하는 방법은 Tutorial 07 참조